# News Agent

The **News Agent** runs in parallel with the Financials Agent after the Planner.

**Pipeline:**
1. Builds a **market-aware search query** (handles US, Indian with suffix, and bare Indian tickers)
2. Calls **Tavily Search API** → 5 most recent articles
3. Passes snippets to **gpt-4o-mini** → 3-bullet business impact summary

## Query logic

| Input ticker | Tavily query built |
|---|---|
| `AAPL` | `AAPL stock news earnings revenue 2025` |
| `RELIANCE.NS` | `RELIANCE NSE India stock news earnings 2025` |
| `TCS.BO` | `TCS BSE India stock news earnings 2025` |
| `SUZLON` | `SUZLON India NSE stock news earnings 2025` |
| `WIPRO` | `WIPRO India NSE stock news earnings 2025` |

> Bare Indian tickers (no suffix) get **India NSE** context added so Tavily returns
> relevant Indian market news rather than unrelated results.

In [ ]:
import os
import nbimporter
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from tools.search import tavily_search

load_dotenv()

_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Well-known Indian tickers that are commonly typed without a suffix.
# When matched, the news query gets India + NSE context added automatically.
_KNOWN_INDIA_TICKERS = {
    "RELIANCE", "TCS", "INFY", "INFOSYS", "WIPRO", "HDFCBANK", "ICICIBANK",
    "SBIN", "AXISBANK", "KOTAKBANK", "LT", "BAJFINANCE", "BHARTIARTL",
    "ASIANPAINT", "MARUTI", "SUNPHARMA", "TITAN", "NESTLEIND", "TATAMOTORS",
    "TATASTEEL", "POWERGRID", "NTPC", "ONGC", "COALINDIA", "ADANIENT",
    "ADANIPORTS", "HINDUNILVR", "ULTRACEMCO", "TECHM", "HCLTECH",
    "SUZLON", "ZOMATO", "PAYTM", "NYKAA", "POLICYBZR", "DMART",
    "IRCTC", "HAL", "BEL", "BHEL", "GAIL", "IOC", "BPCL",
}


def _build_search_query(ticker: str) -> str:
    """
    Build a market-aware Tavily search query.

    Priority:
      1. Explicit .NS suffix  → NSE India query
      2. Explicit .BO suffix  → BSE India query
      3. Bare ticker in known India list → India NSE query
      4. Anything else → standard US query
    """
    t = ticker.upper().strip()

    if t.endswith(".NS"):
        base = t[:-3]
        return f"{base} NSE India stock news earnings results 2025"

    if t.endswith(".BO"):
        base = t[:-3]
        return f"{base} BSE India stock news earnings results 2025"

    if t in _KNOWN_INDIA_TICKERS:
        return f"{t} India NSE stock news earnings results 2025"

    # Default: treat as US ticker
    return f"{t} stock news earnings revenue 2025"


def news_node(state: dict) -> dict:
    """
    Fetch 5 recent articles via Tavily, summarize with gpt-4o-mini.
    Returns: news_articles (raw list) + news_summary (3-bullet string).
    """
    ticker = state["ticker"]
    query  = _build_search_query(ticker)
    print(f"[News] Query: '{query}'")

    articles = tavily_search(query, max_results=5)

    articles_text = "\n\n".join(
        [
            f"Title: {a.get('title', 'N/A')}\n"
            f"URL: {a.get('url', 'N/A')}\n"
            f"Content: {a.get('content', '')[:600]}"
            for a in articles
        ]
    )

    is_india = (
        ticker.upper().endswith((".NS", ".BO"))
        or ticker.upper() in _KNOWN_INDIA_TICKERS
    )
    market_context = "Indian stock (NSE/BSE)" if is_india else "stock"

    prompt = (
        f"Summarize the following news articles about {ticker} {market_context} into "
        f"exactly 3 bullet points. Focus on business impact, earnings, and market sentiment.\n\n"
        f"Articles:\n{articles_text}\n\n"
        f"Return exactly 3 bullet points starting with the • character."
    )

    response = _llm.invoke([HumanMessage(content=prompt)])
    summary  = response.content.strip()

    print(f"[News] Summary ready ({len(summary)} chars)")
    return {"news_articles": articles, "news_summary": summary}


In [ ]:
if __name__ == "__main__":
    # --- Demo: test all 4 ticker formats ---
    cases = [
        {"ticker": "AAPL"},          # US explicit
        {"ticker": "SUZLON"},        # India bare — uses known list
        {"ticker": "RELIANCE.NS"},   # India explicit suffix
        {"ticker": "TCS.BO"},        # India explicit suffix
    ]

    for state in cases:
        t = state["ticker"]
        print(f"\n{'='*55}")
        print(f" Ticker: {t}")
        print(f" Query : {_build_search_query(t)}")
        print(f"{'='*55}")
        result = news_node(state)
        print(result["news_summary"])
